# 🔬 Lab 02: Advanced Chunking Strategies

---

## 🎯 Executive Overview & Production Challenge
Chunking is the single most critical decision in RAG pipeline design. It defines the atomic unit of retrieval.
- **Naive Chunking:** Fixed character or token counts (e.g., 500 characters with 50-character overlap). Slices paragraphs mid-sentence, severs tables in half, and scatters context.
- **Semantic Chunking:** Computes the cosine distance between consecutive sentences. When semantic distance spikes above a threshold, a topic boundary is declared!
- **Parent-Document (Hierarchical) Retrieval:** Indexes small child chunks (e.g. 100 tokens) for crisp vector matching, but passes the broad parent document (e.g. 800 tokens) to the LLM for rich answer synthesis.
- **Sentence-Window Retrieval:** Indexes single target sentences and expands context to $\pm K$ adjacent sentences at query time.

```
       [Sentence 1] ─── Cosine Sim ~ 0.88 ───> Same Semantic Chunk
       [Sentence 2] ─── Cosine Sim ~ 0.84 ───> Same Semantic Chunk
       [Sentence 3] ─── Cosine Sim ~ 0.31 (DROP) ──> [NEW CHUNK BOUNDARY]
       [Sentence 4]
```


In [ ]:
# %pip install sentence_transformers

In [ ]:
import os
import sys
import re
import tempfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# Configure sandboxed cache directories for Torch, HuggingFace, and Matplotlib
temp_dir = tempfile.gettempdir()
os.environ.setdefault("MPLCONFIGDIR", os.path.join(temp_dir, "matplotlib"))
os.environ.setdefault("TORCH_HOME", os.path.join(temp_dir, "torch"))
os.environ.setdefault("HF_HOME", os.path.join(temp_dir, "hf"))
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", os.path.join(temp_dir, "torch_inductor"))

# Self-healing sys.path resolution so rag_universal_client is ALWAYS found
for candidate in [
    Path.cwd(),
    Path.cwd() / "RAG_ContxtEngg_Labs",
    Path(__file__).parent if "__file__" in locals() else None,
    Path.cwd() / "UST" / "RAG_ContxtEngg_Labs",
    Path(r"c:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs")
]:
    if candidate and candidate.exists() and (candidate / "rag_universal_client.py").exists():
        if str(candidate.resolve()) not in sys.path:
            sys.path.insert(0, str(candidate.resolve()))
        break
import rag_universal_client as rc

# Resilient path resolution for sample_documents
DATA_DIR = rc.get_rag_data_dir()
print(f"[*] Verified Data Directory: {DATA_DIR}")

# Load the expanded 10+ page operational manual
with open(DATA_DIR / "turbomachinery_manual.md", "r", encoding="utf-8") as f:
    raw_document = f.read()

# Define alias for cross-compatibility
manual_text = raw_document

print(f"[✓] Loaded turbomachinery_manual.md: {len(raw_document.splitlines())} lines | {len(raw_document)} characters.")


---
## Step 1: Naive Fixed-Size Chunking (The Anti-Pattern)
Let's see what happens when we split text using naive character slicing.


In [ ]:
# Resilient fallback in case Cell 1 was run before variable rename
if "raw_document" not in locals():
    if "manual_text" in locals():
        raw_document = manual_text
    else:
        with open(DATA_DIR / "turbomachinery_manual.md", "r", encoding="utf-8") as f:
            raw_document = f.read()
            manual_text = raw_document

def naive_fixed_chunker(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

naive_chunks = naive_fixed_chunker(raw_document)
print(f"[!] Naive chunker produced {len(naive_chunks)} chunks.")
print("\n--- Sample Naive Chunk Boundary Artifact ---")
print(f"Chunk 2 End:   ...{naive_chunks[1][-60:]}")
print(f"Chunk 3 Start: {naive_chunks[2][:60]}...")


---
## Step 2: Semantic Chunking from Scratch
Semantic chunking calculates the semantic distance between adjacent sentences and cuts chunks only at true topic shifts.


In [ ]:
# Initialize local fast embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def split_into_sentences(text):
    """Split document into clean sentences."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 15]

sentences = split_into_sentences(raw_document)
print(f"[*] Extracted {len(sentences)} individual sentences.")

# Generate embeddings for each sentence
embeddings = embedder.encode(sentences, normalize_embeddings=True)

# Calculate cosine similarity between consecutive sentences
similarities = []
for i in range(len(embeddings) - 1):
    sim = np.dot(embeddings[i], embeddings[i+1])
    similarities.append(float(sim))

# Dynamic threshold: Cut where similarity is below (mean - 0.7 * std)
mean_sim = np.mean(similarities)
std_sim = np.std(similarities)
threshold = mean_sim - (0.7 * std_sim)

print(f"Cosine Similarity Mean: {mean_sim:.3f} | Std: {std_sim:.3f}")
print(f"Dynamic Semantic Boundary Threshold: {threshold:.3f}")


Let's visualize the sentence-by-sentence similarity curve and the identified chunk cut points:


In [ ]:
plt.figure(figsize=(10, 4), dpi=120)
plt.plot(similarities, marker='o', markersize=4, color='#2563eb', label='Consecutive Sentence Similarity')
plt.axhline(threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.2f})')

# Mark split points
split_indices = [i for i, s in enumerate(similarities) if s < threshold]
plt.scatter(split_indices, [similarities[i] for i in split_indices], color='red', s=70, zorder=5, label='Semantic Cut Point')

plt.title('Semantic Distance Transitions across Document Sentences', fontsize=11, fontweight='bold')
plt.xlabel('Sentence Transition Index', fontsize=9)
plt.ylabel('Cosine Similarity', fontsize=9)
plt.legend(loc='lower left', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
def assemble_semantic_chunks(sentences, similarities, threshold):
    chunks = []
    curr_chunk = [sentences[0]]
    for i, sim in enumerate(similarities):
        if sim < threshold:
            chunks.append(" ".join(curr_chunk))
            curr_chunk = [sentences[i+1]]
        else:
            curr_chunk.append(sentences[i+1])
    if curr_chunk:
        chunks.append(" ".join(curr_chunk))
    return chunks

semantic_chunks = assemble_semantic_chunks(sentences, similarities, threshold)
print(f"[✓] Created {len(semantic_chunks)} coherent Semantic Chunks.")
print(f"\n--- Sample Semantic Chunk (Self-Contained Topic) ---")
print(semantic_chunks[1][:300] + "...")


---
## Step 3: Parent-Document (Hierarchical) Retrieval (Small-to-Big)
In high-precision RAG:
1. Embed and index small **Child Chunks** (~100 tokens). They match user queries with razor-sharp similarity.
2. Link each child chunk via `parent_id` to its broad **Parent Document** (~800 tokens).
3. At query time, vector search retrieves the child chunk, but the system passes the parent document to the LLM!


In [ ]:
class HierarchicalChunkStore:
    def __init__(self):
        self.parents = {}
        self.children = []
        self.child_embeddings = None

    def add_document(self, doc_id, text, parent_size=600, child_size=120):
        # Create parent sections
        p_idx = 0
        for p_start in range(0, len(text), parent_size):
            parent_id = f"{doc_id}_P{p_idx}"
            parent_text = text[p_start : p_start + parent_size]
            self.parents[parent_id] = parent_text
            
            # Create child chunks within parent
            for c_start in range(0, len(parent_text), child_size):
                child_id = f"{parent_id}_C{len(self.children)}"
                child_text = parent_text[c_start : c_start + child_size]
                if len(child_text.strip()) > 30:
                    self.children.append({
                        "child_id": child_id,
                        "parent_id": parent_id,
                        "text": child_text
                    })
            p_idx += 1

    def build_index(self, embedder):
        child_texts = [c["text"] for c in self.children]
        self.child_embeddings = embedder.encode(child_texts, normalize_embeddings=True)

    def search(self, query, embedder, top_k=2):
        q_emb = embedder.encode([query], normalize_embeddings=True)[0]
        scores = np.dot(self.child_embeddings, q_emb)
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            child = self.children[idx]
            parent_text = self.parents[child["parent_id"]]
            results.append({
                "score": float(scores[idx]),
                "child_id": child["child_id"],
                "parent_id": child["parent_id"],
                "matched_child_text": child["text"],
                "full_parent_context": parent_text
            })
        return results

store = HierarchicalChunkStore()
store.add_document("MANUAL_2026", raw_document)
store.build_index(embedder)
print(f"[✓] Hierarchical Store: {len(store.parents)} Parents | {len(store.children)} Children indexed.")


In [ ]:
test_query = "What is the primary vent leakage flow limit for dry gas seals?"
matches = store.search(test_query, embedder, top_k=1)
m = matches[0]
print(f"Query: '{test_query}'")
print(f"Cosine Match Score: {m['score']:.4f}")
print(f"\n[1] Small Child Vector Match (Pinpoint accuracy):")
print(f"    >> '{m['matched_child_text']}'")
print(f"\n[2] Parent Context Passed to LLM (No missing facts!):")
print(f"    >> '{m['full_parent_context']}'")


---
## Step 4: Sentence-Window Retrieval
Sentence-window retrieval indexes isolated individual sentences, but attaches a metadata window of $\pm K$ neighboring sentences.


In [ ]:
def sentence_window_search(query, sentences, embedder, window_k=2):
    s_embs = embedder.encode(sentences, normalize_embeddings=True)
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(s_embs, q_emb)
    best_idx = int(np.argmax(scores))
    
    start_idx = max(0, best_idx - window_k)
    end_idx = min(len(sentences), best_idx + window_k + 1)
    
    window_context = " ".join(sentences[start_idx:end_idx])
    return {
        "score": float(scores[best_idx]),
        "best_sentence": sentences[best_idx],
        "window_context": window_context,
        "window_range": (start_idx, end_idx)
    }

win_res = sentence_window_search("What is the mandatory Omanization ratio for engineering staff?", sentences, embedder)
print(f"[✓] Sentence-Window Result (Score: {win_res['score']:.4f}):")
print(f"  Target Sentence: '{win_res['best_sentence']}'")
print(f"  Surrounding Window (+/- 2 sentences):\n  '{win_res['window_context']}'")


---
## 🏁 Production Engineering Takeaways & Exercises
1. **Never use static naive character chunking** for dense enterprise manuals or tables.
2. **Use Parent-Document Retrieval** when your documents have deep explanatory context that child chunks alone cannot capture.
3. **Use Semantic Chunking** when text has natural narrative shifts across paragraphs.
